In [0]:
import pandas as pd
import calendar


In [0]:
# %sql
# CREATE SCHEMA ska_catalog.bronze;

In [0]:
# %sql
# CREATE VOLUME ska_catalog.bronze.bronze_volume

In [0]:
# dbutils.fs.mkdirs("/Volumes/ska_catalog/bronze/bronze_volume/raw_data")

In [0]:
df_sales = spark.read.format("csv")\
    .option("header", "True")\
    .option("inferSchema", "True")\
    .load("/Volumes/ska_catalog/bronze/bronze_volume/raw_data/sales/sales_data.csv")

df_sales.display()


In [0]:
def rename_col(df_sales: object, columns: list) -> object:
    for col in columns:
        df_sales = df_sales.withColumnRenamed(col, col.strip().lower().replace(" ", ""))
    return df_sales  # Moved return outside the loop to process all columns

df_sales = rename_col(df_sales, df_sales.columns)
df_sales.display()

In [0]:

df_sales_pd = pd.read_csv("/Volumes/ska_catalog/bronze/bronze_volume/raw_data/sales/sales_data.csv",header = 0)

df_sales_pd = df_sales_pd.rename(columns = lambda x: x.strip().lower().replace(" ", ""))

df_sales_pd = df_sales_pd.map(lambda x: x.strip().replace("$","").replace(",","") if isinstance(x, str) else x)

df_sales_pd["sale_id"] = df_sales_pd.index + 1

df_sales_pd = df_sales_pd.astype({"segment": object,
"country":object,"product":object,"discountband":object,"unitssold": int
,"manufacturingprice":float,"saleprice":float,"grosssales":float,"discounts":float,"sales":float,"cogs":float,"profit":float,"date": "datetime64[s]","monthnumber":int,"monthname":object,"year":int,"managerid":object,"sale_id":int}, errors="raise")

df_sales_pd.display()

In [0]:
df_sales_pd["unitPrice*sales"] = df_sales_pd['unitssold'] * df_sales_pd['saleprice']

In [0]:
# df_sales_pd["unit*slaes"] = df_sales_pd.apply(lambda row : row['unitssold'] * row['saleprice'], axis = 1)


# df_sales_pd.drop(columns = ['unit*slaes'],inplace = True)

df_sales_pd.display()

In [0]:
# Select distinct product names.
distinct_df = df_sales_pd.set_index('sale_id').drop_duplicates().reset_index()

print(len(distinct_df))
distinct_df.display()

In [0]:
# List all records where discountband is 'High'.
df_filtered_high = df_sales_pd[df_sales_pd['discountband'] == 'High']
df_filtered_high.display()
print(len(df_filtered_high))

In [0]:
# Show the total number of units sold for each segment
df_unit_sold_by_segment = df_sales_pd.groupby('segment').agg({'unitssold':"sum"}).rename(columns = {'unitssold': 'units per segment'}).reset_index()
df_unit_sold_by_segment.display()

In [0]:
# Find all sales made in the year 2022.
df_sales_2022 = df_sales_pd [df_sales_pd['date'].dt.year == 2022]
df_sales_2022.display()
print(len(df_sales_2022))

In [0]:
# Display the product, unitssold, and profit for each sale.
df_sales_pd[["sale_id",'product', 'unitssold', 'profit']].set_index('sale_id').display()

In [0]:
# List all unique managerid values.
df_unique_mng_id = pd.DataFrame({'Manager ID':df_sales_pd['managerid'].unique()})
df_unique_mng_id.display()

# print count of unique values.
df_sales_pd['managerid'].nunique()
df_sales_pd['managerid'].value_counts()


In [0]:
# Get the number of sales per monthname.
df_sales_per_month = df_sales_pd.groupby(df_sales_pd['date'].dt.month_name()).agg({'sale_id':"count"}).rename(columns = {'sale_id': 'sales per month'}).reset_index()

# Converting into categorical column.
df_sales_per_month['date'] = pd.Categorical(df_sales_per_month['date'], categories = list(calendar.month_name[1:]), ordered=True)

df_sales_per_month = df_sales_per_month.sort_values('date').reset_index(drop=True)
df_sales_per_month.display()

In [0]:
# Show all records where profit is greater than 10,000.

df_profit = df_sales_pd[ df_sales_pd['profit']> 10000].sort_values(by = 'profit', ascending = False)
df_profit.display()


In [0]:
# Calculate total sales and profit for each product.
df_sale_per_product = df_sales_pd\
        .groupby('product')\
        .agg(
            {'profit':'sum',
                'sales':'sum'
            })\
        .rename(columns = {'profit':'total_profit','sales':'total_sales'})\
        .reset_index()

In [0]:
# Find the top 5 products with the highest total profit.
df_sale_per_product = df_sales_pd\
        .groupby('product')\
        .agg(
            {'profit':'sum',
                'sales':'sum'
            })\
        .rename(columns = {'profit':'total_profit','sales':'total_sales'})\
        .sort_values(by = 'total_profit', ascending = False)\
        .head(5)\
        .reset_index()
df_sale_per_product.display()

In [0]:
df_unitssold_discount = df_sales_pd\
    .groupby('discountband')\
    .agg({'unitssold':'mean'})\
    .round(2)\
    .reset_index()
df_unitssold_discount.display()


In [0]:
# Extract year and month from the date column.
df_sales_pd['year_new'] = df_sales_pd['date'].dt.year
df_sales_pd['monthnumber_new'] = df_sales_pd['date'].dt.month
df_sales_pd.display()

In [0]:
# Count sales entries per managerid and sort by count.
df_sales_entries = df_sales_pd.groupby('managerid')\
    .agg({'sale_id': 'count'}).reset_index()

df_sales_entries = df_sales_entries.sort_values(by = 'sale_id',ascending = False).display()

In [0]:
# Show the total grosssales per country and year.
df_sales_per_country = df_sales_pd.groupby(by = ['country', 'year_new']).agg({'grosssales': 'sum'}).reset_index()
df_sales_per_country.display()

In [0]:
# List products with average profit greater than 5,000.
df_avg_profits = df_sales_pd.groupby('product')\
    .agg({
        'profit': 'mean'
    }).round(4).reset_index()

df_avg_profits = df_avg_profits[df_avg_profits['profit']>5000]
df_avg_profits.display()

In [0]:
#Find the number of sales where discounts are zero.
#Pandas shape
# Definition: Returns the dimensions of a DataFrame or Series as a tuple (rows, columns).
df_sales_0_discount = df_sales_pd[df_sales_pd['discounts']==0].shape[0]
print(df_sales_0_discount)

In [0]:
# Display the top 3 months with the highest total sales.
df_3top_sales = df_sales_pd\
    .groupby(df_sales_pd['date'].dt.month_name())\
    .agg({'sales':'sum'})\
    .round(2)\
    .rename(columns = {'sales':'total_sales'})\
    .reset_index()

df_3top_sales['date'] = pd.Categorical(df_3top_sales['date'],categories= list(calendar.month_name[1:]),ordered= True)

df_3top_sales.sort_values(by = 'total_sales', ascending = False).reset_index(drop = True).display ()

In [0]:
df_sales_country = df_sales_pd.groupby('country')\
    .agg({'sales': 'sum'})\
    .rename(columns = {'sales':'total_sales'})\
    .sort_values(by = 'total_sales', ascending = False)\
    .reset_index()
df_sales_country[(df_sales_country['country']=='Canada') | (df_sales_country['country']=='France')].display()

df_sales_pd.display()

In [0]:
# Calculate profit margin (profit/sales) for each sale and rank them.
df_sales_pd['profit_margin'] = (df_sales_pd['profit']/df_sales_pd['sales']).round(8)

df_sales_pd['rank'] = df_sales_pd['profit_margin'].rank(method = 'dense', ascending = False)

df_sales_pd = df_sales_pd.sort_values(by = 'rank', ascending = True).reset_index(drop = True)
df_sales_pd.display()


In [0]:
# Calculate profit margin (profit/sales) for each sale and rnk them
df_sales_pd['profit_margin'] = df_sales_pd['profit']/df_sales_pd['sales']

df_sales_pd['rank'] = df_sales_pd['profit_margin'].rank(method = 'dense',ascending = False).reset_index(drop = True)

In [0]:
df_profit_mgr = df_sales_pd.groupby(['managerid',df_sales_pd['year_new']]).agg({'profit':'sum'}).reset_index()
most_profitable_per_year =df_profit_mgr.loc[df_profit_mgr.groupby('year_new')['profit'].idxmax()]
most_profitable_per_year.display()

In [0]:
# Compare monthly sales trends for 'Amarilla' across countries.

df_sales_pd_amarilla = df_sales_pd[ df_sales_pd['product'] == 'Amarilla']

monthly_trend = df_sales_pd_amarilla.groupby(['country','monthname']).agg({'sales':'sum'}).round(4).reset_index()

monthly_trend.sort_values( ['country','monthname'], ascending = [True,True])
monthly_trend.display()

In [0]:
from pandas.api.types import CategoricalDtype
month_order = CategoricalDtype(
    categories=['January', 'February', 'March', 'April','May', 'June', 'July', 'August', 'September','October', 'November','December'],
    ordered=True
)

df_sales_pd['monthname'] = df_sales_pd['monthname'].astype(month_order)
df_sales_pd.info()

In [0]:
# Calculate cumulative sales per country in date order.

df_sales_pd = df_sales_pd.sort_values('date', ascending=True)
df_sales_pd['cum_sales'] = (
    df_sales_pd
    .groupby('country')['sales']
    .transform('cumsum')
)
display(df_sales_pd)

In [0]:
df_top3_segments = df_sales_pd.groupby('segment')\
    .agg({'profit': 'mean'})\
    .rename(columns={'profit': 'avg_profit'})\
    .round(4)\
    .sort_values(by='avg_profit', ascending=False)\
    .head(3)\
    .reset_index()
df_top3_segments.display()

In [0]:
# Calculate monthly sales per product
monthly_sales = df_sales_pd.groupby(['product', 'year_new', 'monthnumber_new'])['sales'].sum().reset_index()

# Sort for consecutive month comparison
monthly_sales = monthly_sales.sort_values(['product', 'year_new', 'monthnumber_new'])

# Identify products with declining sales over consecutive months
monthly_sales['decline'] = monthly_sales.groupby('product')['sales'].diff() < 0

display(monthly_sales)

# Find products with at least one period of consecutive monthly decline
declining_products = monthly_sales.groupby('product')['decline'].any().reset_index()
print(declining_products)
declining_products = declining_products[declining_products['decline']]['product']

print(declining_products)

# Filter monthly_sales for only declining products
declining_sales = monthly_sales[monthly_sales['product'].isin(declining_products)]

display(declining_sales)

In [0]:
df_declining_sales = spark.createDataFrame(declining_sales)
df_declining_sales.write.mode('overwrite').saveAsTable('ska_catalog.bronze.silver_declining_sales')


In [0]:
df_sales_pd.display()

In [0]:

df_sales_pd["unit*sale"] = df_sales_pd.apply(lambda row: row['unitssold'] * row['saleprice'], axis=1)
df_sales_pd.display()